# DIY Copilot
## Lesson: Hello GenAI World

Every coding assistant – GitHub Copilot, Cursor, Claude Code – is, at its core, a program that sends **messages** to a large language model (LLM) and does something with the answer. In this lesson you will make your very first call to an LLM and build the smallest possible "coding assistant".

We proceed in 5 steps:

- Step 1: Set up the environment
- Step 2: Show available models
- Step 3: Configure the model
- Step 4: Set up the prompt
- Step 5: Set up the generation configuration

### Step 1: Set up the environment

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [ ]:
from litellm import completion, get_valid_models

from lib.environment import load_environment

# Load the API key (Colab secret or .env file) and the default models
default_models = load_environment()

### Step 2: Show available models
We use the **Gemini API** via [LiteLLM](https://docs.litellm.ai/). LiteLLM offers one unified (OpenAI-compatible) interface for more than 100 providers, so switching from Gemini to OpenAI, Anthropic or a local model is mostly a matter of changing the model name.

Let's ask the provider which models are available for our API key.

In [ ]:
PROVIDER = "gemini"

valid_models = get_valid_models(check_provider_endpoint=True, custom_llm_provider=PROVIDER)

for model in valid_models:
    print(model)

### Step 3: Configure the model
LiteLLM expects model ids in the format `<provider>/<model>`. Note that the list above contains models for different tasks (text generation, embeddings, images, ...). There is no endpoint telling you which model is meant for which task – you have to check the provider documentation.

In [ ]:
MODEL = "gemini-3.1-flash-lite"
MODEL_ID = f"{PROVIDER}/{MODEL}"

### Step 4: Set up the prompt
A request consists of a list of **messages**. Each message has a **role**:
* `system`: instructions that define the behaviour of the assistant. In a coding assistant, this is where the tool vendor puts its (often very long) instructions.
* `user`: the request of the user.
* `assistant`: previous answers of the model (we will need this later for conversations).

In [ ]:
SYSTEM_PROMPT = "You are a senior Python developer. You answer concisely and always include a short code example."
USER_PROMPT = "How do I check in Python whether two time ranges overlap?"

### Step 5: Set up the generation configuration

In [ ]:
# Set some generation parameters. We will learn more about them in the next lesson.
TEMPERATURE = 1.0
MAX_TOKENS = 1000

In [ ]:
# Print out a summary of the input values / parameters
print(f'SYSTEM PROMPT used:\n {SYSTEM_PROMPT}')
print(f'USER PROMPT used:\n {USER_PROMPT}')
print(f'MODEL used:\n {MODEL_ID} (temperature = {TEMPERATURE}, max_tokens = {MAX_TOKENS})')

### Exercise 01: Call the model
Generate a response by calling the *completion* method from `litellm` (see imports). Use the *model*, *messages* and *generation configuration parameters* as arguments.

>**Note:** Free tier models are sometimes overloaded and answer with an error (HTTP 503). With `num_retries=3`, LiteLLM automatically tries again.

In [ ]:
# TODO: Set up the messages: the system prompt for the role "system" and the user prompt for the role "user"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

# TODO: Generate a response by calling the "completion" method from litellm.
# Use the model, messages and generation configuration parameters as arguments.
response = completion(
    model=MODEL_ID,
    messages=messages,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    num_retries=3,  # retry automatically if the model is temporarily overloaded
)

In [ ]:
# Print the complete response of the model
print('RESPONSE of the model:')
print(response)

### Exercise 02: Inspect the response
Print only the message of the model. What is the `role` of the message?

Then have a look at the `usage` of the response. Every coding assistant counts tokens: they determine the **costs**, the **latency** and how much code fits into the **context window** of the model.

In [ ]:
# TODO: Print the role of the message
print('ROLE of the message:')
print(response.choices[0].message.role)
print()

# TODO: Print the content of the message
print('CONTENT of the message:')
print(response.choices[0].message.content)

In [ ]:
# TODO: Print the number of input (prompt) tokens and output (completion) tokens
print(f"Input tokens:  {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")

### Exercise 03: Shape your assistant with the system prompt
The same model behaves completely differently depending on the system prompt. Coding assistants use this to create different "modes" (e.g. *ask*, *edit*, *review*).

Write two system prompts and compare the answers to the same user prompt:
1. A **mentor** who explains the solution step by step to a junior developer.
2. A **code generator** that answers with Python code only – no explanations, no Markdown.

In [ ]:
# TODO: Define the two system prompts
MENTOR_PROMPT = (
    "You are a patient mentor for junior Python developers. "
    "Explain the solution step by step in simple words before you show the code."
)
CODE_ONLY_PROMPT = (
    "You are a code generator. Answer with valid Python code only. "
    "Do not add explanations and do not wrap the code in Markdown code fences."
)

In [ ]:
# TODO: Call the model with both system prompts and the same user prompt and print the answers
for name, system_prompt in [("MENTOR", MENTOR_PROMPT), ("CODE ONLY", CODE_ONLY_PROMPT)]:
    response = completion(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": USER_PROMPT},
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        num_retries=3,
    )
    print(f"========== {name} ==========")
    print(response.choices[0].message.content)
    print()

### Default models
To avoid configuring the model in every notebook, the following notebooks use the default models returned by `load_environment()`. You can override them by adding the secrets `DEFAULT_GENERATION_MODEL` and `DEFAULT_EMBEDDING_MODEL` (Colab) or the corresponding entries in your `.env` file (local).

In [ ]:
print(f"Default model for generation: {default_models['generation_model']}")
print(f"Default model for embeddings: {default_models['embedding_model']}")